# IL 轨迹预测训练示例

演示 `il` 模块的数据加载与训练流程：
1. 构建 Hydra 风格的 DictConfig
2. 使用 create_dataloaders 创建 DataLoader
3. 检查数据集和 batch 结构
4. 构建模型并执行一次前向传播
5. 调用 train 进行完整训练

In [ ]:
import torch
from pathlib import Path
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
import time

from il.dataset import create_dataloaders
from il.model import ConditionalUNet1D
from il.training.df_train import _batch_to_device, prepare_diffusion_schedule, add_noise
from il.utils.normalizer import BatchNormalizer
import os

def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到项目根目录（未发现 pyproject.toml）")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())

%cd {PROJECT_ROOT}

## 1. 构建配置

il 模块使用 Hydra DictConfig，包含 `data`、`model`、`train`、`loss`、`log` 五个子配置。

`data.data_dir` 应指向包含 TrainingSample pkl 文件的目录。

In [ ]:
conf_dir = (Path.cwd() / "src/il/conf").resolve()
if conf_dir is None:
    raise FileNotFoundError("未找到 Hydra 配置目录 src/il/conf")

with initialize_config_dir(version_base=None, config_dir=str(conf_dir)):
    il_cfg = compose(config_name="config",
        overrides=[
        "training=train_df",
    ],)

# from omegaconf import OmegaConf

data_dir = (Path.cwd() / "data").resolve()
il_cfg.training.data.data_dirs = [f"{data_dir}/20260502_021749/preprocessed"]

log_dir = (Path.cwd() / "log").resolve()
il_cfg.training.log.save_dir = f"{log_dir}/il_tutorial"
il_cfg.training.trainer.checkpoint_path = f"{log_dir}/il_train_df_20260504_231108/best_model.pt"

print(il_cfg.training.trainer.checkpoint_path)
print(OmegaConf.to_yaml(il_cfg, resolve=False))

## 2. 创建 DataLoader

`create_dataloaders` 按文件列表顺序划分 train / val / test，返回三个 DataLoader。

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(il_cfg.training)

print(f"训练集: {len(train_loader.dataset)} 样本, {len(train_loader)} batch")
print(f"验证集: {len(val_loader.dataset)} 样本, {len(val_loader)} batch")
print(f"测试集: {len(test_loader.dataset)} 样本, {len(test_loader)} batch")

## 3. 检查 batch 结构

In [ ]:
batch = next(iter(train_loader))

print("Batch keys 和 shape:")
for key, val in batch.items():
    print(f"  {key:25s}: {val.shape}  dtype={val.dtype}")

print(f"\nvehicle_params 示例 (第一个样本):")
print(f"  [a_max, omega_max, v_max, wheelbase, length, width, height,")
print(f"   front_overhang, rear_overhang, mass, drag_coeff]")
print(f"  {batch['vehicle_params'][0].numpy()}")

## 前向加噪

In [ ]:
batch_norm = BatchNormalizer.from_config(il_cfg.training.data.normalization)
normal_batch = batch_norm.normalize(batch)
device = il_cfg.training.device
normal_batch = _batch_to_device(normal_batch, device)
x = normal_batch["future"]
future_mask = normal_batch["future_mask"]
cond = normal_batch

schedule = prepare_diffusion_schedule(il_cfg.training, device)
T = len(schedule["betas"])
T_max = T - 1

# Same eps for all t so multi-step plots show one consistent forward-diffusion path
eps_shared = torch.randn_like(x)


def forward_xt(x0: torch.Tensor, t_scalar: int) -> torch.Tensor:
    """x_t = sqrt(alphas_cumprod_t) * x_0 + sqrt(1 - alphas_cumprod_t) * eps (same as add_noise, fixed eps)."""
    t_b = torch.full((x0.shape[0],), t_scalar, device=device, dtype=torch.long)
    sqrt_acp = schedule["sqrt_alphas_cumprod"][t_b]
    sqrt_om = schedule["sqrt_one_minus_alphas_cumprod"][t_b]
    while sqrt_acp.ndim < x0.ndim:
        sqrt_acp = sqrt_acp.unsqueeze(-1)
        sqrt_om = sqrt_om.unsqueeze(-1)
    return sqrt_acp * x0 + sqrt_om * eps_shared


step = 100
timesteps_plot = list(range(0, T, step))
if timesteps_plot[-1] != T_max:
    timesteps_plot.append(T_max)

x_noisy_list = [forward_xt(x, t) for t in timesteps_plot]

# Plot: (1) GT only  (2) grid of noisy trajectories at different diffusion steps
import math

import matplotlib.pyplot as plt

idx = 0
xm = future_mask[idx].detach().cpu().numpy()
xy_clean = x[idx, :, :2].detach().cpu().numpy()

fig_gt, ax_gt = plt.subplots(figsize=(6, 5))
ax_gt.plot(xy_clean[xm, 0], xy_clean[xm, 1], "o-", color="tab:green", lw=2, ms=4, label="GT x0")
ax_gt.scatter(xy_clean[xm, 0][0], xy_clean[xm, 1][0], c="black", s=60, zorder=5, marker="s", label="start")
ax_gt.set_title(f"sample {idx}: ground-truth future (ego-local xy)")
ax_gt.set_xlabel("x (m)")
ax_gt.set_ylabel("y (m)")
ax_gt.set_aspect("equal", adjustable="box")
ax_gt.grid(alpha=0.3)
ax_gt.legend(loc="best", fontsize=9)
fig_gt.suptitle("DDPM forward — clean trajectory", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

n_panels = len(timesteps_plot)
ncols = min(3, n_panels)
nrows = math.ceil(n_panels / ncols)
fig_ns, axs_ns = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False, sharex=True, sharey=True)
cmap = plt.cm.plasma

for k, t_show in enumerate(timesteps_plot):
    r, c = divmod(k, ncols)
    ax = axs_ns[r][c]
    xy_noisy = x_noisy_list[k][idx, :, :2].detach().cpu().numpy()
    color = cmap(t_show / max(T_max, 1))
    ax.plot(xy_clean[xm, 0], xy_clean[xm, 1], "--", color="tab:green", lw=1.2, alpha=0.65, label="GT")
    ax.plot(xy_noisy[xm, 0], xy_noisy[xm, 1], "o-", color=color, lw=2, ms=3, label=f"x_t")
    ax.scatter(xy_noisy[xm, 0][0], xy_noisy[xm, 1][0], c="black", s=45, zorder=5, marker="s")
    ax.set_title(f"t = {t_show} / {T_max}")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.3)

for j in range(n_panels, nrows * ncols):
    r, c = divmod(j, ncols)
    axs_ns[r][c].set_visible(False)

for ax in axs_ns[-1]:
    if ax.get_visible():
        ax.set_xlabel("x (m)")
for ax in axs_ns[:, 0]:
    if ax.get_visible():
        ax.set_ylabel("y (m)")

fig_ns.suptitle(
    "Forward diffusion at multiple steps (same ε across t for comparable curves)",
    fontsize=12,
    y=1.02,
)
plt.tight_layout()
plt.show()


In [ ]:
# shapes after forward-noise cell above
len(timesteps_plot), x.shape


# 模型加载

In [ ]:
# 加载 IL 训练模型，进行闭环验证（IL 轨迹预测 + MPC 跟踪）
import numpy as np
from pathlib import Path
from hydra import compose

from il.model import ConditionalUNet1D


def _to_local_coords(points: np.ndarray, origin: np.ndarray, theta: float) -> np.ndarray:
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    delta = points - origin
    local_x = delta[..., 0] * cos_t + delta[..., 1] * sin_t
    local_y = -delta[..., 0] * sin_t + delta[..., 1] * cos_t
    return np.stack([local_x, local_y], axis=-1)


def _normalize_angle(angle: float, ref_theta: float) -> float:
    return float((angle - ref_theta + np.pi) % (2 * np.pi) - np.pi)


def _build_model_inputs(history_buf, obs, info, cfg):
    dc = cfg.data
    H = int(dc.history_len) + 1
    N = int(dc.road_points)
    D = int(dc.num_lane_dividers)

    history = np.asarray(history_buf, dtype=np.float32)
    history_mask = np.ones(H, dtype=np.float32)

    # 保留世界坐标下的 ego 位姿，用于把模型局部预测结果还原到世界坐标
    ego_xy_world = history[-1, :2].copy()
    ego_theta_world = float(history[-1, 2])

    centerline = np.asarray(obs["centerline"], dtype=np.float32)
    left_boundary = np.asarray(obs["left_boundary"], dtype=np.float32)
    right_boundary = np.asarray(obs["right_boundary"], dtype=np.float32)

    lane_dividers = np.zeros((D, N, 2), dtype=np.float32)
    if "lane_dividers" in obs:
        ld_obs = np.asarray(obs["lane_dividers"], dtype=np.float32)
        d_actual = min(D, ld_obs.shape[0])
        lane_dividers[:d_actual] = ld_obs[:d_actual]

    actual_len = int(info.get("actual_length_num", N))
    actual_len = max(0, min(actual_len, N))
    road_mask = np.zeros(N, dtype=np.float32)
    road_mask[:actual_len] = 1.0

    centerline_mask = road_mask.copy()
    left_boundary_mask = road_mask.copy()
    right_boundary_mask = road_mask.copy()
    lane_dividers_mask = np.tile(road_mask[None, :], (D, 1))

    if bool(dc.use_local_coords):
        ego_xy = history[-1, :2].copy()
        ego_theta = float(history[-1, 2])

        history[:, :2] = _to_local_coords(history[:, :2], ego_xy, ego_theta).astype(np.float32)
        for i in range(history.shape[0]):
            history[i, 2] = _normalize_angle(float(history[i, 2]), ego_theta)

        centerline = _to_local_coords(centerline, ego_xy, ego_theta).astype(np.float32)
        left_boundary = _to_local_coords(left_boundary, ego_xy, ego_theta).astype(np.float32)
        right_boundary = _to_local_coords(right_boundary, ego_xy, ego_theta).astype(np.float32)
        for d in range(D):
            lane_dividers[d] = _to_local_coords(lane_dividers[d], ego_xy, ego_theta).astype(np.float32)

    return {
        "history": history,
        "history_mask": history_mask,
        "centerline": centerline,
        "centerline_mask": centerline_mask,
        "left_boundary": left_boundary,
        "left_boundary_mask": left_boundary_mask,
        "right_boundary": right_boundary,
        "right_boundary_mask": right_boundary_mask,
        "lane_dividers": lane_dividers,
        "lane_dividers_mask": lane_dividers_mask,
        "ego_pose_world": np.array([ego_xy_world[0], ego_xy_world[1], ego_theta_world], dtype=np.float32),
    }


def _predict_ref_path(model, model_inputs, cfg, device):
    with torch.no_grad():
        b = {
            k: torch.from_numpy(v).unsqueeze(0).to(device)
            for k, v in model_inputs.items()
            if k != "ego_pose_world"
        }
        pred = model(
            b["history"], b["history_mask"],
            b["centerline"], b["centerline_mask"],
            b["left_boundary"], b["left_boundary_mask"],
            b["right_boundary"], b["right_boundary_mask"],
            b["lane_dividers"], b["lane_dividers_mask"],
        ).squeeze(0).cpu().numpy()  # (F, 4): x,y,heading,v

    pred_xy = pred[:, :2]
    pred_v = pred[:, 3]

    # 若模型在局部坐标预测，需要转回世界坐标供控制器跟踪
    if bool(cfg.data.use_local_coords):
        ex, ey, etheta = [float(x) for x in model_inputs["ego_pose_world"]]
        cos_t, sin_t = np.cos(etheta), np.sin(etheta)
        world_xy = np.zeros_like(pred_xy)
        world_xy[:, 0] = ex + pred_xy[:, 0] * cos_t - pred_xy[:, 1] * sin_t
        world_xy[:, 1] = ey + pred_xy[:, 0] * sin_t + pred_xy[:, 1] * cos_t
        pred_xy = world_xy

    target_speed = float(np.clip(pred_v[0], 0.0, 30.0))
    return pred_xy, target_speed


# 1) 加载 IL checkpoint

ckpt_path = (Path.cwd() / il_cfg.training.trainer.checkpoint_path).resolve()
if not ckpt_path.exists():
    raise FileNotFoundError(f"未找到 checkpoint: {ckpt_path}")

model = ConditionalUNet1D(il_cfg.training).to(torch.device(il_cfg.training.device))
state = torch.load(ckpt_path, map_location=il_cfg.training.device)
model.load_state_dict(state["model_state_dict"])
model.eval()
print(f"已加载模型: {ckpt_path}")


# 模型开环验证（DDPM）

与 `df_train` 一致：`BatchNormalizer` → `prepare_diffusion_schedule` → `sample_ddpm`；ADE/FDE 与轨迹图均在 **反归一化后的物理空间** 计算。


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from il.training.df_train import _batch_to_device, prepare_diffusion_schedule, sample_ddpm
from il.utils.normalizer import BatchNormalizer


def open_loop_visualize_grid_with_dynamics_df(
    model,
    loader,
    cfg,
    n_samples=4,
    seed=0,
    target_speed=None,
):
    """与 ``df_train`` 一致：BatchNormalizer + ``prepare_diffusion_schedule`` + ``sample_ddpm``。"""
    random.seed(seed)
    torch.manual_seed(seed)

    train_cfg = cfg.training
    device = torch.device(train_cfg.device)
    model.eval()

    batch_norm = BatchNormalizer.from_config(train_cfg.data.normalization)
    schedule = prepare_diffusion_schedule(train_cfg, device=device)

    all_ade, all_fde = [], []
    samples = []

    if loader is None:
        raise ValueError("loader 不能为空，请先创建 dataloader")

    for batch in loader:
        b = _batch_to_device(batch, device)
        b.pop("vehicle_params", None)
        if target_speed is not None:
            b["max_v"] = torch.ones_like(b["max_v"]) * float(target_speed)
            b["max_v_mask"] = torch.ones_like(b["max_v_mask"], dtype=b["max_v_mask"].dtype)

        nb = batch_norm.normalize(b)
        cond = nb
        fut = nb["future"]
        shape = (fut.shape[0], fut.shape[1], fut.shape[2])

        with torch.no_grad():
            pred_norm = sample_ddpm(model, cond, schedule, shape)

        pred = batch_norm.inverse_future(pred_norm)
        gt = batch_norm.inverse_future(nb["future"])
        phys = batch_norm.inverse(nb)

        m = b["future_mask"].float()
        disp = torch.norm(pred[..., :2] - gt[..., :2], dim=-1)
        valid = m.sum(dim=-1).clamp(min=1)
        ade_batch = ((disp * m).sum(dim=-1) / valid).detach().cpu().numpy()

        rev = torch.flip(m, dims=[1])
        last_idx = m.size(1) - 1 - torch.argmax(rev, dim=1).long()
        fde_batch = disp.gather(1, last_idx.unsqueeze(1)).squeeze(1).detach().cpu().numpy()

        all_ade.extend(ade_batch.tolist())
        all_fde.extend(fde_batch.tolist())

        pick_ids = list(range(pred.shape[0]))
        random.shuffle(pick_ids)
        fut_m_bool = b["future_mask"].bool()
        for bi in pick_ids:
            if len(samples) >= n_samples:
                break
            samples.append({
                "hist": phys["history"][bi].detach().cpu().numpy(),
                "hist_m": b["history_mask"][bi].detach().cpu().numpy() > 0.5,
                "fut": gt[bi].detach().cpu().numpy(),
                "fut_m": fut_m_bool[bi].detach().cpu().numpy(),
                "pd": pred[bi].detach().cpu().numpy(),
                "cl": phys["centerline"][bi].detach().cpu().numpy(),
                "cl_m": b["centerline_mask"][bi].detach().cpu().numpy() > 0.5,
                "lb": phys["left_boundary"][bi].detach().cpu().numpy(),
                "lb_m": b["left_boundary_mask"][bi].detach().cpu().numpy() > 0.5,
                "rb": phys["right_boundary"][bi].detach().cpu().numpy(),
                "rb_m": b["right_boundary_mask"][bi].detach().cpu().numpy() > 0.5,
                "ade": float(ade_batch[bi]),
                "fde": float(fde_batch[bi]),
                "fut_heading": gt[bi, :, 2].detach().cpu().numpy(),
                "pred_heading": pred[bi, :, 2].detach().cpu().numpy(),
                "fut_speed": gt[bi, :, 3].detach().cpu().numpy(),
                "pred_speed": pred[bi, :, 3].detach().cpu().numpy(),
            })
        if len(samples) >= n_samples:
            break

    if not samples:
        print("没有可用样本用于开环验证")
        return

    # 图1：轨迹 2x2
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    for i in range(4):
        ax = axes[i]
        if i >= len(samples):
            ax.axis("off")
            continue
        s = samples[i]
        if s["cl_m"].any():
            ax.plot(s["cl"][s["cl_m"], 0], s["cl"][s["cl_m"], 1], "--", c="gold", lw=1.5, label="centerline")
        if s["lb_m"].any():
            ax.plot(s["lb"][s["lb_m"], 0], s["lb"][s["lb_m"], 1], "-", c="gray", lw=1.2, label="left boundary")
        if s["rb_m"].any():
            ax.plot(s["rb"][s["rb_m"], 0], s["rb"][s["rb_m"], 1], "-", c="dimgray", lw=1.2, label="right boundary")
        ax.plot(s["hist"][s["hist_m"], 0], s["hist"][s["hist_m"], 1], "o-", c="tab:blue", lw=2, ms=3, label="history")
        ax.plot(s["fut"][s["fut_m"], 0], s["fut"][s["fut_m"], 1], "o-", c="tab:green", lw=2, ms=3, label="future gt")
        ax.plot(s["pd"][:, 0], s["pd"][:, 1], "o-", c="tab:red", lw=2, ms=3, label="DDPM pred")
        ax.scatter(s["hist"][s["hist_m"]][-1, 0], s["hist"][s["hist_m"]][-1, 1], c="black", s=40, marker="x", label="current")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.3)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        ax.set_title(f"Sample #{i+1} | ADE={s['ade']:.3f}, FDE={s['fde']:.3f}")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4)
    fig.suptitle("DDPM Open-loop Trajectory (4 Samples)", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # 图2：速度 / 航向角 2x2
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    axes2 = axes2.flatten()
    for i in range(4):
        ax = axes2[i]
        if i >= len(samples):
            ax.axis("off")
            continue
        s = samples[i]
        fut_m = s["fut_m"]
        t = np.arange(len(s["pred_speed"]))
        t_valid = t[fut_m]

        l1 = ax.plot(t_valid, s["fut_speed"][fut_m], "o-", c="tab:green", lw=1.8, ms=3, label="gt speed")
        l2 = ax.plot(t, s["pred_speed"], "o-", c="tab:red", lw=1.8, ms=3, label="pred speed")
        ax.set_xlabel("future step")
        ax.set_ylabel("speed (m/s)", color="tab:red")
        ax.tick_params(axis="y", labelcolor="tab:red")
        ax.grid(alpha=0.3)

        ax_h = ax.twinx()
        l3 = ax_h.plot(t_valid, s["fut_heading"][fut_m], "--", c="tab:blue", lw=1.6, label="gt heading")
        l4 = ax_h.plot(t, s["pred_heading"], "--", c="purple", lw=1.6, label="pred heading")
        ax_h.set_ylabel("heading (rad)", color="tab:blue")
        ax_h.tick_params(axis="y", labelcolor="tab:blue")

        ax.set_title(f"Sample #{i+1} dynamics")

        lines = l1 + l2 + l3 + l4
        labs = [ln.get_label() for ln in lines]
        ax.legend(lines, labs, loc="best", fontsize=8)

    fig2.suptitle("DDPM Open-loop Dynamics: Speed & Heading (4 Samples)", fontsize=14)
    fig2.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    print("\n=== 开环验证指标（物理空间 xy）===")
    print(f"样本数     : {len(all_ade)}")
    print(f"平均 ADE   : {np.mean(all_ade):.4f}")
    print(f"平均 FDE   : {np.mean(all_fde):.4f}")


open_loop_visualize_grid_with_dynamics_df(model, test_loader, il_cfg, n_samples=4, seed=42, target_speed=None)


In [ ]:
torch.full((10,), 1)